# Workshop 2 · Power BI Dataset Performance · Exercise

![Power BI report mockup](../../../assets/images/powerbi_report_mockup.png)

How this visual helps: it defines the target outcome for the exercise. Each task creates a Databricks object that should make this kind of Power BI report faster, easier to refresh or easier to govern.

In this workshop you build the Day 2 serving layer for Power BI: a monthly aggregate, an incremental-refresh view and a KPI table. This is not a repeat of Day 1 modeling; it is the Power BI dataset performance layer on top of the Gold model.


## Business Scenario

RetailHub already has a governed Gold model from Workshop 1. The Power BI team now needs serving objects that make reports faster, cheaper and easier to refresh.

In this workshop you play the role of a Power BI developer preparing a production-ready dataset contract on top of Databricks. The focus is not new business logic; the focus is selecting the right table shape for Power BI consumption.

## Success Criteria

A correct solution must satisfy:

- `gold.fact_sales_dashboard_monthly` exists at monthly x region x category x channel grain,
- `gold.v_fact_sales_incremental` exposes `order_datetime` for Power BI incremental refresh,
- `gold.kpi_monthly` contains one row per `year_month`,
- monthly totals reconcile to `gold.fact_sales_dashboard` for completed rows,
- incremental refresh window uses a half-open interval,
- the BI contract states source object, grain, model mode and refresh expectation.

Power BI developer lens: each object exists because a report design decision needs it. Do not create serving objects without knowing whether they support Import pages, drill-through, KPI cards or refresh validation.

## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before starting the workshop.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

Day 2 starts from the Gold model created in Workshop 1. This check fails early if the model is not available.


In [ ]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Run Workshop 1 solution before starting Day 2.")
print("[OK] Day 1 Gold model is available.")


### Workshop Validation Helpers

These helper functions keep the task-level assertions short and readable. They are used only for validation, not for the main SQL implementation.


In [ ]:
# -- Validation helpers --
def _table_exists(full_name):
    return spark.catalog.tableExists(full_name)


def _require_table(full_name):
    assert _table_exists(full_name), f"Missing object: {full_name}"


def _columns(full_name):
    return set(spark.table(full_name).columns)


def _require_columns(full_name, required_columns):
    missing = sorted(set(required_columns) - _columns(full_name))
    assert not missing, f"{full_name} missing columns: {missing}"


def _scalar(query):
    row = spark.sql(query).first()
    return row[0] if row is not None else None


def _first(query):
    row = spark.sql(query).first()
    assert row is not None, "Query returned no rows"
    return row


def _print_ok(message):
    print(message)


## Source Baseline

Business context: before building serving objects, capture the size and date range of the W1 dashboard table.

Why Power BI cares: this baseline helps decide whether a report should import detail, use an aggregate, or reserve detail for drill-through.

Required output: row count, order count, min date, max date and completed revenue from `gold.fact_sales_dashboard`.

In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  COUNT(DISTINCT order_id) AS distinct_orders,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM gold.fact_sales_dashboard


## Task 1: Build `gold.fact_sales_dashboard_monthly`

Power BI decision: this object is the Import-friendly aggregate for summary pages.

Business context: executive dashboard visuals usually show trends by month, category, region and channel. They do not need every order line.

Required grain: `year_month x customer_region x category x channel`.

Required measures: `line_rows`, `completed_orders`, `quantity`, `revenue`, `gross_margin`, `margin_rate_pct`.

Implementation hints:

- source: `gold.fact_sales_dashboard`,
- filter to completed rows only,
- include `order_month` as a date version of `year_month`,
- use `COUNT(DISTINCT order_id)` for completed orders,
- protect margin-rate division with `NULLIF`,
- add a table `COMMENT` that explains the grain.

Acceptance criteria: one row per monthly grain key and no detail-level columns such as `line_id`.

Common mistake: grouping by too many columns and recreating a line-grain table under a monthly name.

**Guidance — Task 1**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.fact_sales_dashboard_monthly.
-- Source: gold.fact_sales_dashboard.
-- Filter: completed rows only.
-- Grain: year_month, customer_region, category, channel.
-- Required: COMMENT on the table.


In [ ]:
%sql
SELECT
  COUNT(*) AS aggregate_rows,
  COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS distinct_grain_keys,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.fact_sales_dashboard_monthly


In [ ]:
# -- Validation --
table = f"{GOLD}.fact_sales_dashboard_monthly"
_require_table(table)
_require_columns(table, ["year_month", "order_month", "customer_region", "category", "channel", "line_rows", "completed_orders", "quantity", "revenue", "gross_margin", "margin_rate_pct"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel)) AS duplicate_grain_keys, SUM(CASE WHEN line_rows <= 0 THEN 1 ELSE 0 END) AS invalid_line_rows
FROM {GOLD}.fact_sales_dashboard_monthly
""")
assert row["rows"] > 0, "Task 1 failed: monthly aggregate is empty"
assert row["duplicate_grain_keys"] == 0, f"Task 1 failed: duplicate monthly grain keys = {row['duplicate_grain_keys']}"
assert row["invalid_line_rows"] == 0, f"Task 1 failed: invalid line_rows buckets = {row['invalid_line_rows']}"
_print_ok(f"Task 1 OK: monthly aggregate has {row['rows']} unique grain rows.")


## Task 2: Reconcile Monthly Aggregate to Detail

Power BI decision: an aggregate is acceptable only if it reconciles to the trusted detail source for the same KPI rule.

Business context: if monthly revenue does not match completed detail revenue, report users will see different totals between summary and drill-through pages.

Required output: `detail_revenue`, `aggregate_revenue`, `revenue_diff`.

Implementation hints:

- detail source: `gold.fact_sales_dashboard` filtered to `is_completed`,
- aggregate source: `gold.fact_sales_dashboard_monthly`,
- round final values for display,
- do not compare returned or cancelled rows because Task 1 excluded them.

Acceptance criteria: revenue difference is zero or a clearly explainable rounding difference.

Common mistake: comparing aggregate completed revenue to unfiltered detail revenue.

**Guidance — Task 2**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: return detail_revenue, aggregate_revenue and revenue_diff.
-- detail source: gold.fact_sales_dashboard where is_completed.
-- aggregate source: gold.fact_sales_dashboard_monthly.


In [ ]:
# -- Validation --
row = _first(f"""
WITH detail AS (SELECT ROUND(SUM(line_revenue), 2) AS revenue FROM {GOLD}.fact_sales_dashboard WHERE is_completed), aggregate AS (SELECT ROUND(SUM(revenue), 2) AS revenue FROM {GOLD}.fact_sales_dashboard_monthly)
SELECT ROUND(detail.revenue - aggregate.revenue, 2) AS revenue_diff FROM detail CROSS JOIN aggregate
""")
assert row["revenue_diff"] is not None, "Task 2 failed: revenue diff is null"
assert abs(float(row["revenue_diff"])) < 0.01, f"Task 2 failed: aggregate/detail revenue diff = {row['revenue_diff']}"
_print_ok("Task 2 OK: monthly aggregate reconciles to completed detail revenue.")


## Task 3: Build `gold.v_fact_sales_incremental`

Power BI decision: this object supports drill-through detail and incremental refresh windows.

Business context: Power BI incremental refresh expects a date/time column that can be filtered by `RangeStart` and `RangeEnd`.

Required output: view `gold.v_fact_sales_incremental` with all columns from `gold.fact_sales_dashboard` plus `order_datetime`.

Implementation hints:

- create a view, not a table,
- use `CAST(order_date AS TIMESTAMP) AS order_datetime`,
- keep line grain unchanged,
- do not pre-filter to completed rows unless the BI contract says detail pages never need other statuses.

Acceptance criteria: row count matches `gold.fact_sales_dashboard` and `order_datetime` is available.

Common mistake: creating an aggregate view and then trying to use it for drill-through.

**Guidance — Task 3**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.v_fact_sales_incremental.
-- Include all columns from gold.fact_sales_dashboard.
-- Add order_datetime = CAST(order_date AS TIMESTAMP).


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines,
  MIN(order_datetime) AS min_order_datetime,
  MAX(order_datetime) AS max_order_datetime
FROM gold.v_fact_sales_incremental


In [ ]:
# -- Validation --
table = f"{GOLD}.v_fact_sales_incremental"
_require_table(table)
_require_columns(table, ["line_id", "order_date", "order_datetime", "line_revenue", "is_completed"])
row = _first(f"""
SELECT (SELECT COUNT(*) FROM {GOLD}.fact_sales_dashboard) AS dashboard_rows, COUNT(*) AS incremental_rows, SUM(CASE WHEN order_datetime IS NULL THEN 1 ELSE 0 END) AS null_order_datetime
FROM {GOLD}.v_fact_sales_incremental
""")
assert row["incremental_rows"] == row["dashboard_rows"], f"Task 3 failed: incremental rows {row['incremental_rows']} do not match dashboard rows {row['dashboard_rows']}"
assert row["null_order_datetime"] == 0, f"Task 3 failed: null order_datetime rows = {row['null_order_datetime']}"
_print_ok(f"Task 3 OK: incremental view exposes order_datetime for {row['incremental_rows']} detail rows.")


## Task 4: Test Incremental Refresh Window

Power BI decision: refresh windows must be non-overlapping and predictable.

Business context: Power BI uses `RangeStart` and `RangeEnd` parameters. The safest Databricks SQL pattern is a half-open interval.

Required filter pattern: `order_datetime >= RangeStart AND order_datetime < RangeEnd`.

Implementation hints:

- simulate `RangeStart = 2024-01-01` and `RangeEnd = 2024-02-01`,
- return rows, min/max order date and completed revenue,
- do not use `BETWEEN` because adjacent windows can overlap.

Acceptance criteria: max date is inside the window and the query excludes the boundary at `RangeEnd`.

Common mistake: using `<= RangeEnd`, which can double-count boundary rows across partitions.

**Guidance — Task 4**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: simulate RangeStart = 2024-01-01 and RangeEnd = 2024-02-01.
-- Return rows, min/max order date and completed revenue for that window.


In [ ]:
# -- Validation --
row = _first(f"""
WITH params AS (SELECT TIMESTAMP '2024-01-01 00:00:00' AS RangeStart, TIMESTAMP '2024-02-01 00:00:00' AS RangeEnd), windowed AS (SELECT v.* FROM {GOLD}.v_fact_sales_incremental v CROSS JOIN params p WHERE v.order_datetime >= p.RangeStart AND v.order_datetime < p.RangeEnd)
SELECT COUNT(*) AS rows_in_window, MIN(order_datetime) AS min_order_datetime, MAX(order_datetime) AS max_order_datetime, SUM(CASE WHEN order_datetime >= TIMESTAMP '2024-02-01 00:00:00' THEN 1 ELSE 0 END) AS rows_on_or_after_range_end
FROM windowed
""")
assert row["rows_in_window"] > 0, "Task 4 failed: test refresh window returned no rows"
assert row["rows_on_or_after_range_end"] == 0, "Task 4 failed: half-open interval includes RangeEnd boundary rows"
_print_ok(f"Task 4 OK: half-open incremental window returned {row['rows_in_window']} rows and excludes RangeEnd.")


## Task 5: Build `gold.kpi_monthly`

Power BI decision: this object supports KPI cards and refresh sanity checks.

Business context: KPI cards should not force Power BI to scan line-grain detail for every executive page load.

Required grain: one row per `year_month`.

Required measures: revenue, gross margin, completed orders, returned orders, margin rate %, return rate %, average order value.

Implementation hints:

- source: `gold.fact_sales_dashboard`,
- use completed rows for revenue, margin and completed orders,
- use returned rows for returned orders,
- protect all ratio denominators with `NULLIF`,
- include `order_month` for date relationships or sorting.

Acceptance criteria: one row per month, no duplicate `year_month`, non-null KPI values for active months.

Common mistake: averaging line-level percentages instead of recalculating ratios from numerator and denominator.

**Guidance — Task 5**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace gold.kpi_monthly.
-- Source: gold.fact_sales_dashboard.
-- Grain: one row per year_month.
-- Required: protect divisions with NULLIF.


In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT year_month) AS distinct_months,
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month
FROM gold.kpi_monthly


In [ ]:
# -- Validation --
table = f"{GOLD}.kpi_monthly"
_require_table(table)
_require_columns(table, ["year_month", "order_month", "revenue", "gross_margin", "completed_orders", "returned_orders", "margin_rate_pct", "return_rate_pct", "avg_order_value"])
row = _first(f"""
SELECT COUNT(*) AS rows, COUNT(*) - COUNT(DISTINCT year_month) AS duplicate_months, SUM(CASE WHEN revenue IS NULL OR gross_margin IS NULL OR completed_orders IS NULL THEN 1 ELSE 0 END) AS null_core_kpis, SUM(CASE WHEN margin_rate_pct IS NOT NULL AND (margin_rate_pct < -100 OR margin_rate_pct > 100) THEN 1 ELSE 0 END) AS suspicious_margin_rates
FROM {GOLD}.kpi_monthly
""")
assert row["rows"] > 0, "Task 5 failed: kpi_monthly is empty"
assert row["duplicate_months"] == 0, f"Task 5 failed: duplicate year_month rows = {row['duplicate_months']}"
assert row["null_core_kpis"] == 0, f"Task 5 failed: rows with null core KPIs = {row['null_core_kpis']}"
assert row["suspicious_margin_rates"] == 0, f"Task 5 failed: suspicious margin rate rows = {row['suspicious_margin_rates']}"
_print_ok(f"Task 5 OK: kpi_monthly has {row['rows']} monthly KPI rows with valid grain.")


## Task 6: BI Contract

Power BI decision: every serving object should have a clear use case before report authors connect to it.

Business context: a semantic model may use different objects for summary pages, drill-through and KPI cards.

Required output columns: `object_name`, `grain`, `recommended_mode`, `refresh_expectation`, `primary_use`.

Implementation hints:

- recommend Import for `gold.fact_sales_dashboard_monthly`,
- recommend Import with incremental refresh or controlled DirectQuery for `gold.v_fact_sales_incremental`,
- recommend Import for `gold.kpi_monthly`,
- describe the report page or semantic-model purpose for each object.

Acceptance criteria: a Power BI developer can choose the correct source object without reading the SQL definition.

Common mistake: treating every Databricks table as equally suitable for every report page.

**Guidance — Task 6**

What you need to do: complete the `%sql` TODO cell directly below so it satisfies the required output and acceptance criteria. Then run the validation cell before moving to the next task.

The validation is intentionally strict: it checks the BI contract that a Power BI developer would rely on, not only whether the SQL statement ran.


In [ ]:
%sql
-- TODO: create or replace TEMP VIEW w2_bi_contract.
-- Required columns: object_name, grain, recommended_mode, refresh_expectation, primary_use.
-- Required objects:
--   gold.fact_sales_dashboard_monthly
--   gold.v_fact_sales_incremental
--   gold.kpi_monthly
-- Acceptance: the validation cell confirms all required objects and non-empty fields.


In [ ]:
%sql
SELECT *
FROM w2_bi_contract
ORDER BY object_name


In [ ]:
# -- Validation --
row = _first("""
SELECT COUNT(*) AS rows, COUNT(DISTINCT object_name) AS distinct_objects, SUM(CASE WHEN grain IS NULL OR recommended_mode IS NULL OR refresh_expectation IS NULL OR primary_use IS NULL THEN 1 ELSE 0 END) AS incomplete_rows
FROM w2_bi_contract
""")
expected_objects = {"gold.fact_sales_dashboard_monthly", "gold.v_fact_sales_incremental", "gold.kpi_monthly"}
observed_objects = {r["object_name"] for r in spark.sql("SELECT object_name FROM w2_bi_contract").collect()}
missing = sorted(expected_objects - observed_objects)
assert row["rows"] >= 3, "Task 6 failed: BI contract must contain at least three serving objects"
assert not missing, f"Task 6 failed: BI contract missing objects: {missing}"
assert row["incomplete_rows"] == 0, f"Task 6 failed: incomplete BI contract rows = {row['incomplete_rows']}"
_print_ok("Task 6 OK: BI contract covers the W2 serving objects with grain, mode, refresh expectation and use case.")


## Completion Checklist

Before moving to Day2_02, confirm:

- monthly aggregate has unique grain keys,
- aggregate revenue reconciles to detail completed revenue,
- incremental view has `order_datetime`,
- KPI table has one row per month,
- BI contract can be handed to the Power BI owner.
